# 07 Results summary (preliminary)
Models trained on raw vs trimmed 2019 audio. Re-run as runs finish; 2021 sets appear once evaluated.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from audio_df.helpers.research.runs import collect, eer_table

MODELS = ["a1", "a4", "a2", "d1", "d2", "d3"]
NAMES = {"a1": "A1 LFCC-GMM", "a4": "A4 LFCC-MLP", "a2": "A2 WavLM", "d1": "D1 concat",
         "d2": "D2 one-way attn", "d3": "D3 two-way attn"}
SETS = {"dev": "2019 dev", "eval": "2019 eval", "eval_clean10k": "2019 eval 10k", "eval_mp3_32k": "MP3 32k",
        "eval_noise_10db": "Noise 10 dB", "eval_opus_16k": "Opus 16k", "eval_resample_8k": "Resample 8k",
        "itw": "In-the-Wild", "asvspoof2021_eval": "2021 LA eval", "asvspoof2021_hidden": "2021 LA hidden"}
RAW, TRIM = "#86b6ef", "#1c5cab"  # blue 250 / 550: one hue, validated as an ordered pair
INK, MUTED, RULE = "#0b0b0b", "#52514e", "#d9d8d4"
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": RULE,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

table = eer_table(collect("../runs", MODELS))
table = table[[s for s in SETS if s in table]]
table.rename(columns=SETS).round(2)

## Raw-trained vs trimmed-trained, per test set
EER %, lower is better. Light dot = trained on raw audio, dark dot = trained on trimmed.

In [ ]:
tests = [s for s in ["eval", "itw", "eval_noise_10db", "eval_mp3_32k", "asvspoof2021_eval", "asvspoof2021_hidden"]
         if s in table]
fig, axes = plt.subplots(1, len(tests), figsize=(3.2 * len(tests), 3.6), sharey=True)
y = np.arange(len(MODELS))[::-1]
for ax, t in zip(axes, tests):
    raw = [table[t].get((m, "raw"), np.nan) for m in MODELS]
    trim = [table[t].get((m, "trimmed"), np.nan) for m in MODELS]
    ax.hlines(y, raw, trim, color=RULE, lw=2, zorder=1)
    ax.scatter(raw, y, s=70, color=RAW, edgecolor="#fcfcfb", lw=2, zorder=2, label="raw-trained")
    ax.scatter(trim, y, s=70, color=TRIM, edgecolor="#fcfcfb", lw=2, zorder=3, label="trimmed-trained")
    for yi, v in zip(y, trim):
        if not np.isnan(v):
            ax.annotate(f"{v:.1f}", (v, yi), xytext=(0, 8), textcoords="offset points", ha="center",
                        fontsize=8, color=MUTED)
    ax.set_title(SETS[t], fontsize=10, loc="left")
    ax.set_xlim(0, max(55, np.nanmax(raw + trim) + 5))
    ax.grid(axis="x", color=RULE, lw=0.6)
    ax.set_axisbelow(True)
    ax.set_xlabel("EER %")
axes[0].set_yticks(y, [NAMES[m] for m in MODELS])
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=2, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

## Trimmed-trained models, every test set
EER %; darker = worse.

In [ ]:
trimmed = table.xs("trimmed", level="train").reindex([m for m in MODELS if (m, "trimmed") in table.index])
fig, ax = plt.subplots(figsize=(1.0 * trimmed.shape[1] + 2, 0.5 * len(trimmed) + 1.2))
ax.imshow(trimmed.to_numpy(), cmap=BLUES, vmin=0, vmax=60, aspect="auto")
for (i, j), v in np.ndenumerate(trimmed.to_numpy()):
    ax.text(j, i, "–" if np.isnan(v) else f"{v:.1f}", ha="center", va="center", fontsize=9,
            color="#fcfcfb" if v > 30 else INK)
ax.set_xticks(range(trimmed.shape[1]), [SETS[c] for c in trimmed.columns], rotation=35, ha="right")
ax.set_yticks(range(len(trimmed)), [NAMES[m] for m in trimmed.index])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.tight_layout()
plt.show()